https://mp.weixin.qq.com/s/xNKdTl5cUPnpVe3OQ3wXKg

```python
from vllm import LLM, SamplingParams
import torch
import torch.profiler as profiler
import os
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
if __name__ == "__main__":
    with profiler.profile(
        activities=[profiler.ProfilerActivity.CPU, profiler.ProfilerActivity.CUDA],
        record_shapes=True,          # 记录张量形状
        profile_memory=True,         # 记录内存分配/释放
        with_stack=False              # close 调用栈
        ) as prof:
        model_name = "/home/kaiyuan/models/Qwen2.5-7B-Instruct" 
        llm = LLM(model=model_name, dtype='float16', tensor_parallel_size=4)
        prompts = [
        "Hello, my name is",
        "The capital of France is",
        "The future of AI is",
        "Please introduce vLLM framework"
        ]
    # 设置采样参数
        sampling_params = SamplingParams(
        temperature=0.8,  # 控制生成文本的随机性，值越高越随机
        top_p=0.95,  # 控制采样范围，值越高生成文本越多样化
        max_tokens=50,  # 生成的最大 token 数量
        n=1
        )
        outputs = llm.generate(prompts, sampling_params)
    prof.export_chrome_trace("trace.json")

```

In [ ]:
!pip install vllm

In [ ]:
from vllm import LLM, SamplingParams
import torch
import torch.profiler as profiler
import os

os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"


with profiler.profile(
    activities=[profiler.ProfilerActivity.CPU, profiler.ProfilerActivity.CUDA],
    record_shapes=True,        # 记录张量形状
    profile_memory=True,       # 记录内存分配/释放
    with_stack=False           # 关闭调用栈
) as prof:
    
    # --- 关键修改 ---
    # 不再使用本地文件路径
    # 直接使用Hugging Face Hub上的模型标识符
    model_name = "RedHatAI/Qwen2.5-VL-3B-Instruct-FP8-dynamic"
    model_name = "Qwen/Qwen2.5-7B-Instruct"
    
    print(f"Loading model '{model_name}'... (Will download if not in cache)")
    
    llm = LLM(model=model_name, dtype='float16', tensor_parallel_size=4)
    
    prompts = [
        "Hello, my name is",
        "The capital of France is",
        "The future of AI is",
        "Please introduce vLLM framework"
    ]
    
    # 设置采样参数
    sampling_params = SamplingParams(
        temperature=0.8,   # 控制生成文本的随机性，值越高越随机
        top_p=0.95,      # 控制采样范围，值越高生成文本越多样化
        max_tokens=50,   # 生成的最大 token 数量
        n=1
    )
    
    print("Generating outputs...")
    outputs = llm.generate(prompts, sampling_params)
    print("Generation complete.")
    
prof.export_chrome_trace("trace.json")
print("Profiler trace saved to trace.json")

In [ ]:
# from google.colab import runtime

# # This command will disconnect the notebook and fully terminate the runtime.
# runtime.unassign()

https://github.com/vllm-project/vllm/blob/main/examples/offline_inference/simple_profiling.py

https://docs.vllm.ai/en/latest/contributing/profiling.html?h=

In [ ]:

import os
import time

from vllm import LLM, SamplingParams

# enable torch profiler, can also be set on cmd line
os.environ["VLLM_TORCH_PROFILER_DIR"] = "./vllm_profile"

os.environ["VLLM_TORCH_PROFILER_RECORD_SHAPES"] = "1"
os.environ["VLLM_TORCH_PROFILER_WITH_PROFILE_MEMORY"] = "1"
os.environ["VLLM_TORCH_PROFILER_WITH_STACK"] = "1"
os.environ["VLLM_TORCH_PROFILER_WITH_FLOPS"] = "1"

os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

# Sample prompts.
prompts = [
    "Hello, my name is",
    "The president of the United States is",
    "The capital of France is",
    "The future of AI is",
]
# Create a sampling params object.
sampling_params = SamplingParams(temperature=0.8, top_p=0.95)


def main():
    # Create an LLM.
    llm = LLM(model="facebook/opt-125m", tensor_parallel_size=1)

    llm.start_profile()

    # Generate texts from the prompts. The output is a list of RequestOutput
    # objects that contain the prompt, generated text, and other information.
    outputs = llm.generate(prompts, sampling_params)

    llm.stop_profile()

    # Print the outputs.
    print("-" * 50)
    for output in outputs:
        prompt = output.prompt
        generated_text = output.outputs[0].text
        print(f"Prompt: {prompt!r}\nGenerated text: {generated_text!r}")
        print("-" * 50)

    # Add a buffer to wait for profiler in the background process
    # (in case MP is on) to finish writing profiling output.
    time.sleep(10)


# if __name__ == "__main__":
#     main()

main()


apt update
apt install -y --no-install-recommends gnupg
echo "deb http://developer.download.nvidia.com/devtools/repos/ubuntu$(source /etc/lsb-release; echo "$DISTRIB_RELEASE" | tr -d .)/$(dpkg --print-architecture) /" | tee /etc/apt/sources.list.d/nvidia-devtools.list
apt-key adv --fetch-keys http://developer.download.nvidia.com/compute/cuda/repos/ubuntu1804/x86_64/7fa2af80.pub
apt update
apt install nsight-systems-cli


VLLM_WORKER_MULTIPROC_METHOD=spawn nsys profile -o report.nsys-rep --trace-fork-before-exec=true --cuda-graph-trace=node python run.py

In [ ]:

import os
import time

from vllm import LLM, SamplingParams

# enable torch profiler, can also be set on cmd line
# os.environ["VLLM_TORCH_PROFILER_DIR"] = "./vllm_profile"

# os.environ["VLLM_TORCH_PROFILER_RECORD_SHAPES"] = "1"
# os.environ["VLLM_TORCH_PROFILER_WITH_PROFILE_MEMORY"] = "1"
# os.environ["VLLM_TORCH_PROFILER_WITH_STACK"] = "1"
# os.environ["VLLM_TORCH_PROFILER_WITH_FLOPS"] = "1"

# os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

# Sample prompts.
prompts = [
    "Hello, my name is",
    "The president of the United States is",
    "The capital of France is",
    "The future of AI is",
]
# Create a sampling params object.
sampling_params = SamplingParams(temperature=0.8, top_p=0.95)


def main():
    # Create an LLM.
    llm = LLM(model="Qwen/Qwen2.5-7B-Instruct", tensor_parallel_size=1)

    # llm.start_profile()

    # Generate texts from the prompts. The output is a list of RequestOutput
    # objects that contain the prompt, generated text, and other information.
    outputs = llm.generate(prompts, sampling_params)

    # llm.stop_profile()

    # Print the outputs.
    print("-" * 50)
    for output in outputs:
        prompt = output.prompt
        generated_text = output.outputs[0].text
        print(f"Prompt: {prompt!r}\nGenerated text: {generated_text!r}")
        print("-" * 50)

    # Add a buffer to wait for profiler in the background process
    # (in case MP is on) to finish writing profiling output.
    time.sleep(10)


if __name__ == "__main__":
    main()

# main()